In [1]:
# Exchange rate: nothing to do here. fx_monthly.csv is already made (data/processed/fx_monthly.csv).

import glob
import pandas as pd

rows = []
files = sorted(glob.glob('../data/raw/pms_reports/*.xlsx'))

# 1. Loop through raw Excel reports
for f in files:
    if '~$' in f:  # skip temporary files created while a workbook is open in Excel
        continue

    sheets = pd.read_excel(f, sheet_name=None, header=None)  # every sheet in the file

    # 2. Find the sheet containing the 'State' table
    ws = None
    for sh in sheets.values():
        if (sh[0].astype(str).str.strip() == 'State').any():
            ws = sh
            break

    if ws is None:
        print('No State table found in', f)
        continue

    # 3. Extract header dates and the national AVERAGE row
    hdr = ws.index[ws[0].astype(str).str.strip() == 'State'][0]
    dates = pd.to_datetime(ws.loc[hdr, [1, 2, 3]]).tolist()  # same month last year, last month, this month
    avg = ws[ws[0].astype(str).str.strip().str.upper() == 'AVERAGE'].iloc[0]
    report_month = dates[2]                                   # the month this report is for

    # 4. Append each month's price record with its source report month
    for col, dt in zip([1, 2, 3], dates):
        rows.append({
            'month': dt.to_period('M').to_timestamp(),
            'pms_price': float(avg[col]),
            'report_month': report_month,
        })

# 5. Deduplicate using the newest report for each month and sort chronologically
pms = (
    pd.DataFrame(rows)
    .sort_values('report_month')  # oldest report first, so keep='last' keeps the newest
    .drop_duplicates('month', keep='last')
    .sort_values('month')
    [['month', 'pms_price']]
)

# 6. Check coverage (January 2023 to May 2026) and export
expected = pd.date_range('2023-01-01', '2026-05-01', freq='MS')
missing = [d.strftime('%Y-%m') for d in expected if d not in set(pms['month'])]
print('Rows:', len(pms), '| missing months:', missing)
print(pms[pms['month'] >= '2023-01-01'])  # spot check: Oct 2023 is about 630.6 and Oct 2024 about 1184.8

pms.to_csv('../data/processed/pms_monthly.csv', index=False)

Rows: 40 | missing months: ['2023-05']
        month    pms_price
81 2023-01-01   257.118725
48 2023-02-01   263.755657
66 2023-03-01   264.289560
36 2023-04-01   254.056231
60 2023-06-01   545.827388
6  2023-07-01   600.347518
39 2023-08-01   626.699840
0  2023-09-01   626.212176
69 2023-10-01   630.634378
30 2023-11-01   648.933552
27 2023-12-01   671.858205
57 2024-01-01   668.296551
51 2024-02-01   679.364483
33 2024-03-01   696.787227
15 2024-04-01   701.244425
21 2024-05-01   769.617031
75 2024-06-01   750.172275
18 2024-07-01   770.541884
24 2024-08-01   830.455009
78 2024-09-01  1030.463388
72 2024-10-01  1184.825423
12 2024-11-01  1214.167452
42 2024-12-01  1189.121238
54 2025-01-01  1258.344614
45 2025-02-01  1245.801134
63 2025-03-01  1261.645767
3  2025-04-01  1239.327111
9  2025-05-01  1027.757378
19 2025-06-01  1037.659429
25 2025-07-01  1024.993006
79 2025-08-01   988.253380
73 2025-09-01   970.587966
13 2025-10-01  1052.311547
43 2025-11-01  1061.350017
55 2025-12-01  1